> ## 🚀 Initial Setup
>
> ---
>
> ### 1️⃣ Trino Database
>
> You can connect to your **own Trino database** using your credentials.
> ---
>
> ### 2️⃣ Docker Image (TPC-H Database)
>
> You can also test with our **pre-built Trino TPC-H database** available on **Docker Hub**. This image will spin up five total containers, one each for Postgres, MySQL, Mongo, Cassandra, and Trino. The first four are used as data sources, and the fifth will be connected to by PyDough as a SQL engine and read data from the first four.
>
> #### 📋 Requirements
> - Make sure you have **Docker Desktop** installed and running.
>
> #### 📦 Pull and Run the Container
> ```bash
> docker run -d --name [CONTAINER_NAME] \
>   -e POSTGRES_USER="host.docker.internal" \
>   -e POSTGRES_USER=[USER] \
>   -e POSTGRES_PASSWORD=[PASSWORD] \
>   -e POSTGRES_PORT=5432 \
>   -e MYSQL_HOST="host.docker.internal" \
>   -e MYSQL_USER=[USER] \
>   -e MYSQL_PASSWORD=[PASSWORD] \
>   -e MYSQL_PORT=3306 \
>   -e MONGO_HOST="host.docker.internal" \
>   -e MONGO_USER=[USER] \
>   -e MONGO_PASSWORD=[PASSWORD] \
>   -e MONGO_PORT=27017 \
>   -e CASSANDRA_HOST="host.docker.internal" \
>   -e CASSANDRA_DC="datacenter1" \
>   -e CASSANDRA_PORT=9042 \
>   -p 8080:5432 bodoai1/pydough-postgres-tpch:latest
> ```
> - Replace `[CONTAINER_NAME]` with your preferred container name.  
> - Replace the various `[USER]` and `[PASSWORD]` with your preferred usernames/passwords for the various sub-containers.
>
> *(Make sure the various ports used are available and not being used by another container.)* 
>
> #### Deleting the container and image
> Once the tests have finished you can stop the container and delete it with the image using the following docker commands:
>```bash
>   docker stop [CONTAINER_NAME]
>   docker rm [CONTAINER_NAME]
>   docker rmi bodoai1/pydough-trino
>```

> ## 🔌 Installing Trino Connector
>
> Make sure to have the **`trino`** package installed:
>
> ---
>
> - **If you're working inside the repo**:
> ```bash
> pip install -e ".[trino]"
> ```
>
> - **Or install the connector directly**:
> ```bash
> pip install trino
> ```


> ## Importing Required Libraries
>
> ---

In [1]:
# Import libraries
import pydough
import trino
import datetime
import os

In [6]:
# Metadata setup
pydough.active_session.load_metadata_graph("../../tests/test_metadata/trino_graphs.json", "TPCH")

# Setup Trino connection
pydough.active_session.connect_database("trino", 
    host="127.0.0.1",
    port=8080,
    user="root",
)


DatabaseContext(connection=<pydough.database_connectors.database_connector.DatabaseConnection object at 0x1196bf5f0>, dialect=<DatabaseDialect.TRINO: 'trino'>)

> ## ✨ Enabling PyDough's Jupyter Magic Commands
>
> ---
>
> This step loads the **`pydough.jupyter_extensions`** module, which adds custom magic commands (like `%%pydough`) to your notebook.
>
> ---
>
> ### 📌 What These Magic Commands Do
> - **Write PyDough directly** in notebook cells using:
>   ```python
>   %%pydough
>   ```
> - **Automatically render** query results inside the notebook.
>
> ---
>
> ### 💻 How It Works
> This is a **Jupyter-specific feature** — the `%load_ext` command dynamically loads these extensions into your **current notebook session**:
> ```python
> %load_ext pydough.jupyter_extensions
> ```


In [7]:
%load_ext pydough.jupyter_extensions

> ## 📊 Running TPC-H Query 1 with PyDough in Trino
>
> ---
>
> This cell runs **TPC-H Query 1** using **PyDough**.
>
> ---
>
> ### 📝 What the Query Does
> - **Computes summary statistics**: sums, averages, and counts for orders.  
> - **Groups by**: `return_flag` and `line_status`.  
> - **Filters by**: a shipping date cutoff.  
>
> ---
>
> ### 📤 Output
> - `pydough.to_df(output)` converts the result to a **Pandas DataFrame**.  
> - This makes it easy to inspect and analyze results directly in Python.  
>
> ---
>


In [ ]:
%%pydough
# TPCH Q1
output = (lines.WHERE((ship_date <= datetime.date(1998, 12, 1)))
        .PARTITION(name="groups", by=(return_flag, status))
        .CALCULATE(
            L_RETURNFLAG=return_flag,
            L_LINESTATUS=status,
            SUM_QTY=SUM(lines.quantity),
            SUM_BASE_PRICE=SUM(lines.extended_price),
            SUM_DISC_PRICE=SUM(lines.extended_price * (1 - lines.discount)),
            SUM_CHARGE=SUM(
                lines.extended_price * (1 - lines.discount) * (1 + lines.tax)
            ),
            AVG_QTY=AVG(lines.quantity),
            AVG_PRICE=AVG(lines.extended_price),
            AVG_DISC=AVG(lines.discount),
            COUNT_ORDER=COUNT(lines),
        )
        .ORDER_BY(L_RETURNFLAG.ASC(), L_LINESTATUS.ASC())
)

pydough.to_df(output)